# Standardization vs Normalization

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We compare Z-score standardization with Min-Max normalization on EEG feature vectors.

## What this notebook does

Reshapes the EEG data to 2D, applies StandardScaler and MinMaxScaler, and prints the resulting statistics.

## What you should expect to see

- Three histograms showing the distribution before and after each scaling method
- Standardized data has mean ~0 and std ~1
- Normalized data is bounded between 0 and 1

## Key parameters

| Parameter | Value |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| scaler | StandardScaler, MinMaxScaler |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Apply standardization and normalization


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_2d = X.reshape(n_trials, n_channels * n_samples)

scaler = StandardScaler()
X_std = scaler.fit_transform(X_2d)

normalizer = MinMaxScaler()
X_norm = normalizer.fit_transform(X_2d)

print(f'Original shape: {X_2d.shape}')
print(f'Raw mean: {X_2d.mean():.4f}, std: {X_2d.std():.4f}')
print(f'Standardized mean: {X_std.mean():.4f}, std: {X_std.std():.4f}')
print(f'Normalized min: {X_norm.min():.4f}, max: {X_norm.max():.4f}')


## 5. Interactive plot

**What to look for:**

- Raw data has a wide range of values centered near zero
- Standardized data is centered at 0 with unit variance
- Normalized data is squeezed into [0, 1]


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

raw_flat = X_2d[:, :1000].flatten()
std_flat = X_std[:, :1000].flatten()
norm_flat = X_norm[:, :1000].flatten()

fig = make_subplots(rows=3, cols=1, subplot_titles=(
    'Raw feature distribution (first 1000 features)',
    'Standardized distribution (Z-score)',
    'Normalized distribution (Min-Max, 0-1)'))

fig.add_trace(go.Histogram(x=raw_flat, nbinsx=100, marker_color='steelblue', opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=std_flat, nbinsx=100, marker_color='orange', opacity=0.7), row=2, col=1)
fig.add_trace(go.Histogram(x=norm_flat, nbinsx=100, marker_color='green', opacity=0.7), row=3, col=1)

fig.update_xaxes(title_text='Value', row=3, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_yaxes(title_text='Count', row=3, col=1)
fig.update_layout(height=900, showlegend=False, title_text='Standardization vs Normalization')
fig.show()


## What did we learn?

- Standardization (Z-score) centers data at 0 with std 1, preserving the shape of the distribution
- Normalization (Min-Max) scales data to [0, 1], sensitive to outliers
- Choice depends on the downstream algorithm: many ML models prefer standardization
